Connected to .venv (Python 3.13.13)

In [ ]:
import mlflow
from dotenv import load_dotenv

load_dotenv(override=True)

import polars as pl


df = pl.read_parquet("https://minio.lab.sspcloud.fr/projet-formation/diffusion/funathon/2026/project2/generation_None_temp08.parquet")
print(df)

df.head(10)
df.height

n_classes = df["code"].n_unique()
print(n_classes)

## 3.1
import numpy as np
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, random_state=1, test_size=0.3)
test, validation = train_test_split(test, random_state=1, test_size=0.5)


X_train, y_train = train["label"].to_numpy(), train["code"].to_numpy()
X_val, y_val = validation["label"].to_numpy(), validation["code"].to_numpy()
X_test, y_test = test["label"].to_numpy(), test["code"].to_numpy()

print(f"Train: {len(train)} | Val: {len(validation)} | Test: {len(test)}")

## 3.2 Encode the Labels
from sklearn.preprocessing import LabelEncoder as LE

le=LE()
le.fit(train['code'].to_numpy())

# Make sure, all codes are contained in test data
np.unique(y_train).size

## 3.3
# Import the ValueEncoder class from the 
# torchTextClassifiers.value_encoder subpackage, 
# then create a value_encoder object by passing your fitted 
# label_encoder as an argument.
from torchTextClassifiers.value_encoder import ValueEncoder

ve = ValueEncoder(label_encoder=le)

# You are going to use a WordPieceTokenizer from the 
# torchTextClassifiers package (see the documentation). 
# Train a WordPieceTokenizer from scratch on the training labels 
# with vocab_size=5000 and output_dim=10. Use the fitted tokenizer 
# to tokenize one observations (with the tokenize method), 
# and inspect the result to understand how the text is split.
import torchTextClassifiers
print(dir(torchTextClassifiers))
## 4.3 Tokenizer
from torchTextClassifiers.tokenizers import WordPieceTokenizer

tn = WordPieceTokenizer(vocab_size=5000, output_dim=10)
#WordPieceTokenizer(vocab_size=5000, output_dim=10)

tn.train(X_train)

print("Output tensor size:", tn.tokenize(X_train[0]).input_ids.shape)
print("Vocabulary size:", tn.vocab_size)

print(tn.tokenize(X_train[1]))
print(X_train[1])


# Look at an example of tokenization
print("Raw text", X_train[0])
print(
    "Tokens id:",
    tn.tokenize(X_train[1]).input_ids.squeeze(0)
)
print(
    "Tokens:",
    tn.tokenizer.convert_ids_to_tokens(
        tn.tokenize(X_train[1]).input_ids.squeeze(0)
    )
)

from torchTextClassifiers import ModelConfig, TrainingConfig, torchTextClassifiers

model_config = ModelConfig(96, 311)
tTC = torchTextClassifiers(tn, model_config, ve)

TConfig = TrainingConfig(lr=1e-3, batch_size=128, num_epochs=1, patience_early_stopping=5)

shape: (70_000, 3)
┌───────┬─────────────────────────────────┬─────────────────────────────────┐
│ code  ┆ name                            ┆ label                           │
│ ---   ┆ ---                             ┆ ---                             │
│ str   ┆ str                             ┆ str                             │
╞═══════╪═════════════════════════════════╪═════════════════════════════════╡
│ 01.11 ┆ Growing of cereals, other than… ┆ Pulses cultivation for market   │
│ 01.11 ┆ Growing of cereals, other than… ┆ Legume crop production activit… │
│ 01.11 ┆ Growing of cereals, other than… ┆ Broad bean farming operations   │
│ 01.11 ┆ Growing of cereals, other than… ┆ Chickpea harvesting and proces… │
│ 01.11 ┆ Growing of cereals, other than… ┆ Production of dried beans and … │
│ …     ┆ …                               ┆ …                               │
│ 82.10 ┆ Office administrative and supp… ┆ Business document preparation … │
│ 82.10 ┆ Office administrative and supp… ┆ L

2026-05-27 14:56:25 - torchTextClassifiers.model.model - 🔹 No categorical variable network provided; using only text embeddings.





Output tensor size: torch.Size([1, 10])
Vocabulary size: 5000
TokenizerOutput(input_ids=tensor([[   3, 1918, 1987,   93,  524,  670,    2,    1,    1,    1]]), attention_mask=tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0]]), offset_mapping=None, word_ids=None)
Habitat relocation and enhancement projects
Raw text Long-haul freight services by motor vehicle
Tokens id: tensor([   3, 1918, 1987,   93,  524,  670,    2,    1,    1,    1])
Tokens: ['[SEP]', 'habitat', 'relocation', 'and', 'enhancement', 'projects', '[CLS]', '[PAD]', '[PAD]', '[PAD]']


In [ ]:
mlflow.set_experiment("funathon-2026-project2")
mlflow.pytorch.autolog()

with mlflow.start_run() as run:
    # This should take approximately 1-2mn
    tTC.train(
        X_train,
        y_train,
        training_config=TConfig,
        X_val=X_val,
        y_val=y_val,
        verbose=True,
    )

    mlflow.log_artifacts(
        training_config.save_path,   # local folder produced by ttc.train()
        artifact_path="model_artifacts",
    )

🏃 View run bold-wasp-813 at: https://user-thomasstat-mlflow.user.lab.sspcloud.fr/#/experiments/1/runs/0fa3ed5fac604af199975b62f4acdc60
🧪 View experiment at: https://user-thomasstat-mlflow.user.lab.sspcloud.fr/#/experiments/1


AssertionError: Y must be a list of lists for ragged multilabel classification.

In [ ]:
model_config = ModelConfig(96, 311)
tTC = torchTextClassifiers(tn, model_config, ve,)

TConfig = TrainingConfig(lr=1e-3, batch_size=128, num_epochs=1, patience_early_stopping=5,)

2026-05-27 14:58:35 - torchTextClassifiers.model.model - 🔹 No categorical variable network provided; using only text embeddings.


In [ ]:
mlflow.set_experiment("funathon-2026-project2")
mlflow.pytorch.autolog()

with mlflow.start_run() as run:
    # This should take approximately 1-2mn
    tTC.train(
        X_train,
        y_train,
        training_config=TConfig,
        X_val=X_val,
        y_val=y_val,
        verbose=True,
    )

    mlflow.log_artifacts(
        training_config.save_path,   # local folder produced by ttc.train()
        artifact_path="model_artifacts",
    )

🏃 View run beautiful-tern-524 at: https://user-thomasstat-mlflow.user.lab.sspcloud.fr/#/experiments/1/runs/82df533e091041c49992d10ca61650b2
🧪 View experiment at: https://user-thomasstat-mlflow.user.lab.sspcloud.fr/#/experiments/1


AssertionError: Y must be a list of lists for ragged multilabel classification.

In [ ]:
model_config = ModelConfig(embedding_dim=96, num_classes=n_classes,)
tTC = torchTextClassifiers(tn, model_config, ve,)

TConfig = TrainingConfig(lr=1e-3, batch_size=128, num_epochs=1, patience_early_stopping=5,)

Restarted .venv (Python 3.13.13)

In [ ]:
import mlflow
from dotenv import load_dotenv

load_dotenv(override=True)

import polars as pl


df = pl.read_parquet("https://minio.lab.sspcloud.fr/projet-formation/diffusion/funathon/2026/project2/generation_None_temp08.parquet")
print(df)

df.head(10)
df.height

n_classes = df["code"].n_unique()
print(n_classes)

## 3.1
import numpy as np
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, random_state=1, test_size=0.3)
test, validation = train_test_split(test, random_state=1, test_size=0.5)


X_train, y_train = train["label"].to_numpy(), train["code"].to_numpy()
X_val, y_val = validation["label"].to_numpy(), validation["code"].to_numpy()
X_test, y_test = test["label"].to_numpy(), test["code"].to_numpy()

print(f"Train: {len(train)} | Val: {len(validation)} | Test: {len(test)}")

## 3.2 Encode the Labels
from sklearn.preprocessing import LabelEncoder as LE

le=LE()
le.fit(train['code'].to_numpy())

# Make sure, all codes are contained in test data
np.unique(y_train).size

## 3.3
# Import the ValueEncoder class from the 
# torchTextClassifiers.value_encoder subpackage, 
# then create a value_encoder object by passing your fitted 
# label_encoder as an argument.
from torchTextClassifiers.value_encoder import ValueEncoder

ve = ValueEncoder(label_encoder=le)

# You are going to use a WordPieceTokenizer from the 
# torchTextClassifiers package (see the documentation). 
# Train a WordPieceTokenizer from scratch on the training labels 
# with vocab_size=5000 and output_dim=10. Use the fitted tokenizer 
# to tokenize one observations (with the tokenize method), 
# and inspect the result to understand how the text is split.
import torchTextClassifiers
print(dir(torchTextClassifiers))
## 4.3 Tokenizer
from torchTextClassifiers.tokenizers import WordPieceTokenizer

tn = WordPieceTokenizer(vocab_size=5000, output_dim=10)
#WordPieceTokenizer(vocab_size=5000, output_dim=10)

tn.train(X_train)

print("Output tensor size:", tn.tokenize(X_train[0]).input_ids.shape)
print("Vocabulary size:", tn.vocab_size)

print(tn.tokenize(X_train[1]))
print(X_train[1])


# Look at an example of tokenization
print("Raw text", X_train[0])
print(
    "Tokens id:",
    tn.tokenize(X_train[1]).input_ids.squeeze(0)
)
print(
    "Tokens:",
    tn.tokenizer.convert_ids_to_tokens(
        tn.tokenize(X_train[1]).input_ids.squeeze(0)
    )
)

from torchTextClassifiers import ModelConfig, TrainingConfig, torchTextClassifiers

shape: (70_000, 3)
┌───────┬─────────────────────────────────┬─────────────────────────────────┐
│ code  ┆ name                            ┆ label                           │
│ ---   ┆ ---                             ┆ ---                             │
│ str   ┆ str                             ┆ str                             │
╞═══════╪═════════════════════════════════╪═════════════════════════════════╡
│ 01.11 ┆ Growing of cereals, other than… ┆ Pulses cultivation for market   │
│ 01.11 ┆ Growing of cereals, other than… ┆ Legume crop production activit… │
│ 01.11 ┆ Growing of cereals, other than… ┆ Broad bean farming operations   │
│ 01.11 ┆ Growing of cereals, other than… ┆ Chickpea harvesting and proces… │
│ 01.11 ┆ Growing of cereals, other than… ┆ Production of dried beans and … │
│ …     ┆ …                               ┆ …                               │
│ 82.10 ┆ Office administrative and supp… ┆ Business document preparation … │
│ 82.10 ┆ Office administrative and supp… ┆ L

In [ ]:
model_config = ModelConfig(embedding_dim=96, num_classes=n_classes,)
tTC = torchTextClassifiers(tn, model_config, ve,)

TConfig = TrainingConfig(lr=1e-3, batch_size=128, num_epochs=1, patience_early_stopping=5,)

2026-05-27 15:00:37 - torchTextClassifiers.model.model - 🔹 No categorical variable network provided; using only text embeddings.


In [ ]:
mlflow.set_experiment("funathon-2026-project2")
mlflow.pytorch.autolog()

with mlflow.start_run() as run:
    # This should take approximately 1-2mn
    tTC.train(
        X_train,
        y_train,
        training_config=TConfig,
        X_val=X_val,
        y_val=y_val,
        verbose=True,
    )

    mlflow.log_artifacts(
        training_config.save_path,   # local folder produced by ttc.train()
        artifact_path="model_artifacts",
    )

🏃 View run abundant-hen-175 at: https://user-thomasstat-mlflow.user.lab.sspcloud.fr/#/experiments/1/runs/69295cf71910426b85fbeff09400318f
🧪 View experiment at: https://user-thomasstat-mlflow.user.lab.sspcloud.fr/#/experiments/1


AssertionError: Y must be a list of lists for ragged multilabel classification.

In [ ]:
import mlflow
from dotenv import load_dotenv

load_dotenv(override=True)

import polars as pl


df = pl.read_parquet("https://minio.lab.sspcloud.fr/projet-formation/diffusion/funathon/2026/project2/generation_None_temp08.parquet")
print(df)

df.head(10)
df.height

n_classes = df["code"].n_unique()
print(n_classes)

## 3.1
import numpy as np
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, random_state=1, test_size=0.3)
test, validation = train_test_split(test, random_state=1, test_size=0.5)


X_train, y_train = train["label"].to_numpy(), train["code"].to_numpy()
X_val, y_val = validation["label"].to_numpy(), validation["code"].to_numpy()
X_test, y_test = test["label"].to_numpy(), test["code"].to_numpy()

print(f"Train: {len(train)} | Val: {len(validation)} | Test: {len(test)}")

## 3.2 Encode the Labels
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
le.fit(train['code'].to_numpy())

# Make sure, all codes are contained in test data
np.unique(y_train).size

## 3.3
# Import the ValueEncoder class from the 
# torchTextClassifiers.value_encoder subpackage, 
# then create a value_encoder object by passing your fitted 
# label_encoder as an argument.
from torchTextClassifiers.value_encoder import ValueEncoder

ve = ValueEncoder(label_encoder=le)

# You are going to use a WordPieceTokenizer from the 
# torchTextClassifiers package (see the documentation). 
# Train a WordPieceTokenizer from scratch on the training labels 
# with vocab_size=5000 and output_dim=10. Use the fitted tokenizer 
# to tokenize one observations (with the tokenize method), 
# and inspect the result to understand how the text is split.
import torchTextClassifiers
print(dir(torchTextClassifiers))
## 4.3 Tokenizer
from torchTextClassifiers.tokenizers import WordPieceTokenizer

tn = WordPieceTokenizer(vocab_size=5000, output_dim=10)
#WordPieceTokenizer(vocab_size=5000, output_dim=10)

tn.train(X_train)

print("Output tensor size:", tn.tokenize(X_train[0]).input_ids.shape)
print("Vocabulary size:", tn.vocab_size)

print(tn.tokenize(X_train[1]))
print(X_train[1])


# Look at an example of tokenization
print("Raw text", X_train[0])
print(
    "Tokens id:",
    tn.tokenize(X_train[1]).input_ids.squeeze(0)
)
print(
    "Tokens:",
    tn.tokenizer.convert_ids_to_tokens(
        tn.tokenize(X_train[1]).input_ids.squeeze(0)
    )
)

from torchTextClassifiers import ModelConfig, TrainingConfig, torchTextClassifiers

shape: (70_000, 3)
┌───────┬─────────────────────────────────┬─────────────────────────────────┐
│ code  ┆ name                            ┆ label                           │
│ ---   ┆ ---                             ┆ ---                             │
│ str   ┆ str                             ┆ str                             │
╞═══════╪═════════════════════════════════╪═════════════════════════════════╡
│ 01.11 ┆ Growing of cereals, other than… ┆ Pulses cultivation for market   │
│ 01.11 ┆ Growing of cereals, other than… ┆ Legume crop production activit… │
│ 01.11 ┆ Growing of cereals, other than… ┆ Broad bean farming operations   │
│ 01.11 ┆ Growing of cereals, other than… ┆ Chickpea harvesting and proces… │
│ 01.11 ┆ Growing of cereals, other than… ┆ Production of dried beans and … │
│ …     ┆ …                               ┆ …                               │
│ 82.10 ┆ Office administrative and supp… ┆ Business document preparation … │
│ 82.10 ┆ Office administrative and supp… ┆ L

In [ ]:
model_config = ModelConfig(embedding_dim=96, num_classes=n_classes,)
tTC = torchTextClassifiers(tn, model_config, ve,)

TConfig = TrainingConfig(lr=1e-3, batch_size=128, num_epochs=1, patience_early_stopping=5,)

2026-05-27 15:03:11 - torchTextClassifiers.model.model - 🔹 No categorical variable network provided; using only text embeddings.


In [ ]:
mlflow.set_experiment("funathon-2026-project2")
mlflow.pytorch.autolog()

with mlflow.start_run() as run:
    # This should take approximately 1-2mn
    tTC.train(
        X_train,
        y_train,
        training_config=TConfig,
        X_val=X_val,
        y_val=y_val,
        verbose=True,
    )

    mlflow.log_artifacts(
        training_config.save_path,   # local folder produced by ttc.train()
        artifact_path="model_artifacts",
    )

🏃 View run orderly-loon-553 at: https://user-thomasstat-mlflow.user.lab.sspcloud.fr/#/experiments/1/runs/80d82f72588c49adafb1a88a3e344dc2
🧪 View experiment at: https://user-thomasstat-mlflow.user.lab.sspcloud.fr/#/experiments/1


AssertionError: Y must be a list of lists for ragged multilabel classification.

In [ ]:
import mlflow
from dotenv import load_dotenv

load_dotenv(override=True)

import polars as pl


df = pl.read_parquet("https://minio.lab.sspcloud.fr/projet-formation/diffusion/funathon/2026/project2/generation_None_temp08.parquet")
print(df)

df.head(10)
df.height

n_classes = df["code"].n_unique()
print(n_classes)

## 3.1
import numpy as np
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, random_state=1, test_size=0.3)
test, validation = train_test_split(test, random_state=1, test_size=0.5)


X_train, y_train = train["label"].to_numpy(), train["code"].to_numpy()
X_val, y_val = validation["label"].to_numpy(), validation["code"].to_numpy()
X_test, y_test = test["label"].to_numpy(), test["code"].to_numpy()

print(f"Train: {len(train)} | Val: {len(validation)} | Test: {len(test)}")

## 3.2 Encode the Labels
from sklearn.preprocessing import LabelEncoder

encoder=LabelEncoder()
encoder.fit(train['code'].to_numpy())

# Make sure, all codes are contained in test data
np.unique(y_train).size

## 3.3
# Import the ValueEncoder class from the 
# torchTextClassifiers.value_encoder subpackage, 
# then create a value_encoder object by passing your fitted 
# label_encoder as an argument.
from torchTextClassifiers.value_encoder import ValueEncoder

value_encoder = ValueEncoder(label_encoder=encoder)

# You are going to use a WordPieceTokenizer from the 
# torchTextClassifiers package (see the documentation). 
# Train a WordPieceTokenizer from scratch on the training labels 
# with vocab_size=5000 and output_dim=10. Use the fitted tokenizer 
# to tokenize one observations (with the tokenize method), 
# and inspect the result to understand how the text is split.
import torchTextClassifiers
print(dir(torchTextClassifiers))
## 4.3 Tokenizer
from torchTextClassifiers.tokenizers import WordPieceTokenizer

tokenizer = WordPieceTokenizer(vocab_size=5000, output_dim=10)
# WordPieceTokenizer(vocab_size=5000, output_dim=10)

tokenizer.train(X_train)

print("Output tensor size:", tokenizer.tokenize(X_train[0]).input_ids.shape)
print("Vocabulary size:", tokenizer.vocab_size)

print(tokenizer.tokenize(X_train[1]))
print(X_train[1])


# Look at an example of tokenization
print("Raw text", X_train[0])
print(
    "Tokens id:",
    tokenizer.tokenize(X_train[1]).input_ids.squeeze(0)
)
print(
    "Tokens:",
    tokenizer.tokenizer.convert_ids_to_tokens(
        tokenizer.tokenize(X_train[1]).input_ids.squeeze(0)
    )
)

from torchTextClassifiers import ModelConfig, TrainingConfig, torchTextClassifiers

shape: (70_000, 3)
┌───────┬─────────────────────────────────┬─────────────────────────────────┐
│ code  ┆ name                            ┆ label                           │
│ ---   ┆ ---                             ┆ ---                             │
│ str   ┆ str                             ┆ str                             │
╞═══════╪═════════════════════════════════╪═════════════════════════════════╡
│ 01.11 ┆ Growing of cereals, other than… ┆ Pulses cultivation for market   │
│ 01.11 ┆ Growing of cereals, other than… ┆ Legume crop production activit… │
│ 01.11 ┆ Growing of cereals, other than… ┆ Broad bean farming operations   │
│ 01.11 ┆ Growing of cereals, other than… ┆ Chickpea harvesting and proces… │
│ 01.11 ┆ Growing of cereals, other than… ┆ Production of dried beans and … │
│ …     ┆ …                               ┆ …                               │
│ 82.10 ┆ Office administrative and supp… ┆ Business document preparation … │
│ 82.10 ┆ Office administrative and supp… ┆ L

In [ ]:
model_config = ModelConfig(embedding_dim=96, num_classes=n_classes,)
ttc = torchTextClassifiers(tokenizer, model_config, value_encoder,)

training_config = TrainingConfig(lr=5e-4, batch_size=128, num_epochs=1, patience_early_stopping=5,)

2026-05-27 15:08:46 - torchTextClassifiers.model.model - 🔹 No categorical variable network provided; using only text embeddings.


In [ ]:
mlflow.set_experiment("funathon-2026-project2")
mlflow.pytorch.autolog()

with mlflow.start_run() as run:
    # This should take approximately 1-2mn
    ttc.train(
        X_train,
        y_train,
        training_config=training_config,
        X_val=X_val,
        y_val=y_val,
        verbose=True,
    )

    mlflow.log_artifacts(
        training_config.save_path,   # local folder produced by ttc.train()
        artifact_path="model_artifacts",
    )

🏃 View run brawny-lamb-325 at: https://user-thomasstat-mlflow.user.lab.sspcloud.fr/#/experiments/1/runs/1f37f58c49024382804de31d20d761bc
🧪 View experiment at: https://user-thomasstat-mlflow.user.lab.sspcloud.fr/#/experiments/1


AssertionError: Y must be a list of lists for ragged multilabel classification.

In [ ]:
import mlflow
from dotenv import load_dotenv

load_dotenv(override=True)

import polars as pl


df = pl.read_parquet("https://minio.lab.sspcloud.fr/projet-formation/diffusion/funathon/2026/project2/generation_None_temp08.parquet")
print(df)

df.head(10)
df.height

n_classes = df["code"].n_unique()
print(n_classes)

## 3.1
import numpy as np
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, random_state=1, test_size=0.3)
test, validation = train_test_split(test, random_state=1, test_size=0.5)


X_train, y_train = train["label"].to_numpy(), train["code"].to_numpy()
X_val, y_val = validation["label"].to_numpy(), validation["code"].to_numpy()
X_test, y_test = test["label"].to_numpy(), test["code"].to_numpy()

print(f"Train: {len(train)} | Val: {len(validation)} | Test: {len(test)}")

## 3.2 Encode the Labels
from sklearn.preprocessing import LabelEncoder

encoder=LabelEncoder()
encoder.fit(train['code'].to_numpy())

# Make sure, all codes are contained in test data
np.unique(y_train).size

## 3.3
# Import the ValueEncoder class from the 
# torchTextClassifiers.value_encoder subpackage, 
# then create a value_encoder object by passing your fitted 
# label_encoder as an argument.
from torchTextClassifiers.value_encoder import ValueEncoder

value_encoder = ValueEncoder(label_encoder=encoder)

# You are going to use a WordPieceTokenizer from the 
# torchTextClassifiers package (see the documentation). 
# Train a WordPieceTokenizer from scratch on the training labels 
# with vocab_size=5000 and output_dim=10. Use the fitted tokenizer 
# to tokenize one observations (with the tokenize method), 
# and inspect the result to understand how the text is split.
import torchTextClassifiers
print(dir(torchTextClassifiers))
## 4.3 Tokenizer
from torchTextClassifiers.tokenizers import WordPieceTokenizer

tokenizer = WordPieceTokenizer(vocab_size=5000, output_dim=10)
# WordPieceTokenizer(vocab_size=5000, output_dim=10)

tokenizer.train(X_train)

print("Output tensor size:", tokenizer.tokenize(X_train[0]).input_ids.shape)
print("Vocabulary size:", tokenizer.vocab_size)

print(tokenizer.tokenize(X_train[1]))
print(X_train[1])


# Look at an example of tokenization
print("Raw text", X_train[0])
print(
    "Tokens id:",
    tokenizer.tokenize(X_train[1]).input_ids.squeeze(0)
)
print(
    "Tokens:",
    tokenizer.tokenizer.convert_ids_to_tokens(
        tokenizer.tokenize(X_train[1]).input_ids.squeeze(0)
    )
)

from torchTextClassifiers import ModelConfig, TrainingConfig, torchTextClassifiers

shape: (70_000, 3)
┌───────┬─────────────────────────────────┬─────────────────────────────────┐
│ code  ┆ name                            ┆ label                           │
│ ---   ┆ ---                             ┆ ---                             │
│ str   ┆ str                             ┆ str                             │
╞═══════╪═════════════════════════════════╪═════════════════════════════════╡
│ 01.11 ┆ Growing of cereals, other than… ┆ Pulses cultivation for market   │
│ 01.11 ┆ Growing of cereals, other than… ┆ Legume crop production activit… │
│ 01.11 ┆ Growing of cereals, other than… ┆ Broad bean farming operations   │
│ 01.11 ┆ Growing of cereals, other than… ┆ Chickpea harvesting and proces… │
│ 01.11 ┆ Growing of cereals, other than… ┆ Production of dried beans and … │
│ …     ┆ …                               ┆ …                               │
│ 82.10 ┆ Office administrative and supp… ┆ Business document preparation … │
│ 82.10 ┆ Office administrative and supp… ┆ L

In [ ]:
model_config = ModelConfig(embedding_dim=96, num_classes=n_classes,)

ttc = torchTextClassifiers(
    tokenizer=tokenizer,
    model_config=model_config,
    value_encoder=value_encoder,
)


training_config = TrainingConfig(lr=5e-4, batch_size=128, num_epochs=1, patience_early_stopping=5,)

2026-05-28 07:07:14 - torchTextClassifiers.model.model - 🔹 No categorical variable network provided; using only text embeddings.


In [ ]:
mlflow.set_experiment("funathon-2026-project2")
mlflow.pytorch.autolog()

with mlflow.start_run() as run:
    # This should take approximately 1-2mn
    ttc.train(
        X_train,
        y_train,
        training_config=training_config,
        X_val=X_val,
        y_val=y_val,
        verbose=True,
    )

    mlflow.log_artifacts(
        training_config.save_path,   # local folder produced by ttc.train()
        artifact_path="model_artifacts",
    )

2026-05-28 07:07:21 - torchTextClassifiers.torchTextClassifiers - Starting training process...
2026-05-28 07:07:21 - torchTextClassifiers.torchTextClassifiers - Running on: cpu
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
/home/onyxia/funathon-project2/.venv/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the 

┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model       │ TextClassificationModel │  510 K │ train │     0 │
│ 1 │ loss        │ CrossEntropyLoss        │      0 │ train │     0 │
│ 2 │ accuracy_fn │ MulticlassAccuracy      │      0 │ train │     0 │
└───┴─────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 510 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 510 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 8                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/onyxia/funathon-project2/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/data.py:79: Trying to
infer the `batch_size` from an ambiguous collection. The batch size we found is 128. To avoid any miscalculations, 
use `self.log(..., batch_size=batch_size)`.

/home/onyxia/funathon-project2/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/data.py:79: Trying to
infer the `batch_size` from an ambiguous collection. The batch size we found is 104. To avoid any miscalculations, 
use `self.log(..., batch_size=batch_size)`.

/home/onyxia/funathon-project2/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/data.py:79: Trying to
infer the `batch_size` from an ambiguous collection. The batch size we found is 4. To avoid any miscalculations, 
use `self.log(..., batch_size=batch_size)`.

`Trainer.fit` stopped: `max_epochs=1` reached.


2026/05/28 07:08:18 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/05/28 07:08:18 WARNING mlflow.utils.requirements_utils: Found torch version (2.6.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.6.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/28 07:08:18 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/onyxia/funathon-project2
2026/05/28 07:08:18 INFO mlflow.utils.environment: Detected uv project at /home/onyxia/funathon-project

🏃 View run fun-boar-963 at: https://user-thomasstat-mlflow.user.lab.sspcloud.fr/#/experiments/1/runs/93a319532069451482268aee75282929
🧪 View experiment at: https://user-thomasstat-mlflow.user.lab.sspcloud.fr/#/experiments/1


In [ ]:
#| label: load-from-run
#| code-overflow: scroll
#| output: true
local_dir = mlflow.artifacts.download_artifacts(
    f"runs:/{run.info.run_id}/model_artifacts"
)

# Rebuild the torchTextClassifiers object from the downloaded files
ttc_loaded = torchTextClassifiers.load(local_dir)

2026-05-28 07:48:20 - torchTextClassifiers.model.model - 🔹 No categorical variable network provided; using only text embeddings.
2026-05-28 07:48:21 - torchTextClassifiers.torchTextClassifiers - Model checkpoint loaded from /tmp/tmpokd37s07/model_artifacts/model_checkpoint.ckpt
2026-05-28 07:48:21 - torchTextClassifiers.torchTextClassifiers - Model loaded successfully from /tmp/tmpokd37s07/model_artifacts


In [ ]:
print(X_test[1:10])

['On-farm processing of spice products'
 'Real property leasing and maintenance'
 'Provision of interpreting for conferences'
 'Drywall installation and repair' 'Full-service lodging establishments'
 'Property leasing and brokerage services'
 'Whisky, brandy and gin manufacturing'
 'Commercial intermediation – non-specialized wholesale'
 'Roof waterproofing and weather protection']


In [ ]:
print(y_test[1:10])

['01.28' '68.20' '74.30' '43.31' '55.10' '68.32' '11.01' '46.19' '43.41']


In [ ]:
example_texts = X_test[1:10]
ttc_loaded.predict(np.array(example_texts), top_k=5,explain_with_captum=True)

{'prediction': array([['68.20', '49.41', '62.10', '43.21', '32.13'],
        ['68.20', '70.20', '66.30', '95.31', '96.22'],
        ['68.20', '66.30', '70.20', '85.59', '96.22'],
        ['95.31', '43.22', '43.21', '62.10', '73.30'],
        ['68.20', '55.20', '49.41', '70.20', '55.10'],
        ['68.20', '70.20', '62.20', '82.10', '66.30'],
        ['68.20', '73.30', '66.30', '95.31', '62.10'],
        ['68.20', '85.59', '70.20', '46.19', '82.10'],
        ['70.20', '68.20', '62.20', '96.22', '62.10']], dtype=object),
 'confidence': tensor([[0.0800, 0.0400, 0.0300, 0.0300, 0.0300],
         [0.5700, 0.0900, 0.0200, 0.0200, 0.0200],
         [0.3000, 0.0400, 0.0400, 0.0300, 0.0200],
         [0.1300, 0.0600, 0.0500, 0.0400, 0.0400],
         [0.1500, 0.0600, 0.0300, 0.0300, 0.0300],
         [0.2100, 0.1400, 0.0500, 0.0500, 0.0300],
         [0.0500, 0.0400, 0.0400, 0.0300, 0.0200],
         [0.1900, 0.0300, 0.0300, 0.0300, 0.0200],
         [0.1200, 0.0900, 0.0500, 0.0500, 0.0400]]),
